## Imports

In [ ]:
import pandas as pd

df_election = pd.read_csv('src/election.csv', sep=';')
df_population = pd.read_csv('src/population.csv', sep=';', dtype={'GEO': str})
df_conjugality = pd.read_csv('src/conjugality.csv', sep=';', dtype={'GEO': str})
df_household = pd.read_csv('src/household.csv', sep=';', dtype={'GEO': str})


def convert_values_to_rates(df):
    columns = [col for col in df.columns if col != 'POPULATION_TOTAL']
    df = df[columns].div(df['POPULATION_TOTAL'], axis=0) * 100
    return df

df_cities = df_population[['GEO', 'CITY']]
df_cities = df_cities.drop_duplicates()
df_cities = df_cities.drop_duplicates(subset=['CITY']).set_index('CITY')
df_cities = df_cities.rename(columns={'GEO': 'CODE_INSEE'})

## Transformation des dataframes en 1 ligne par commune

### DF population

In [ ]:
display(df_population.sort_values(by='CITY'))


df_population_sex = df_population[
    (df_population['AGE'] == '15 ans ou plus') & (df_population['SOCIO_PROF_CAT'] == 'Total')
].pivot_table(
    index='CITY',
    columns='SEX',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={'Femme': 'RATE_SEX_WOMEN', 'Homme': 'RATE_SEX_MEN', 'Total': 'POPULATION_TOTAL'})

df_population_sex = convert_values_to_rates(df_population_sex)


df_population_spc = df_population[
    (df_population['AGE'] == '15 ans ou plus') & (df_population['SEX'] == 'Total')
].pivot_table(
    index='CITY',
    columns='SOCIO_PROF_CAT',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Agriculteurs': 'RATE_SPC_FARMERS',
    'Artisans, commerçants et chefs d’entreprise': 'RATE_SPC_BUSINESS',
    'Cadres et professions intellectuelles supérieures': 'RATE_SPC_EXECUTIVES',
    'Professions intermédiaires': 'RATE_SPC_INTERMEDIATES',
    'Employés': 'RATE_SPC_EMPLOYEES',
    'Ouvriers': 'RATE_SPC_WORKERS',
    'Autres inactifs': 'RATE_SPC_INACTIVES',
    'Retraités': 'RATE_SPC_RETIRED',
    'Etudiants ou élèves': 'RATE_SPC_STUDENTS',
    'Total': 'POPULATION_TOTAL'
})

display(df_population_spc)

df_population_spc = convert_values_to_rates(df_population_spc)


df_population_final = pd.concat([df_population_sex, df_population_spc], axis=1)

display(df_population_final)


### DF conjugality

In [ ]:
display(df_conjugality.sort_values(by='CITY'))


df_conjugality_civil_status = df_conjugality[
    (df_conjugality['AGE'] == '15 ans ou plus') & (df_conjugality['COUPLE'] == 'Total')
].pivot_table(
    index='CITY',
    columns='CIVIL_STATUS',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Marié': 'RATE_MARRIED', 'Pacsé': 'RATE_CIVIL_PARTNER', 'En concubinage, union libre': 'RATE_LIVEIN_PARTNER', 'Divorcé': 'RATE_DIVORCED', 'Célibataire': 'RATE_SINGLE', 'Veuf': 'RATE_WIDOWED', 'Total': 'POPULATION_TOTAL'
})

df_conjugality_civil_status = convert_values_to_rates(df_conjugality_civil_status)


df_conjugality_age = df_conjugality[
    (df_conjugality['COUPLE'] == 'Total') & (df_conjugality['CIVIL_STATUS'] == 'Total')
].pivot_table(
    index='CITY',
    columns='AGE',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'De 15 à 24 ans ': 'RATE_AGE_15_24', 'De 25 à 39 ans': 'RATE_AGE_25_39', 'De 40 à 54 ans': 'RATE_AGE_40_54', 'De 55 à 64 ans': 'RATE_AGE_55_64', 'De 65 à 79 ans': 'RATE_AGE_65_79', '80 ans ou plus': 'RATE_AGE_80+', '15 ans ou plus': 'POPULATION_TOTAL'
})

df_conjugality_age = convert_values_to_rates(df_conjugality_age)


df_conjugality_final = pd.concat([df_conjugality_age, df_conjugality_civil_status], axis=1)

display(df_conjugality_final)

### DF household

In [ ]:
df_household = df_household.drop(['PCS'], axis=1)

display(df_household.sort_values(by='CITY'))


df_household = df_household.pivot_table(
    index='CITY',
    columns='TPH',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Ménage à une personne': 'RATE_ONE_PERSON',
    'Homme seul': 'RATE_ONE_MAN',
    'Femme seule': 'RATE_ONE_WOMAN',
    'Ménage sans famille à plusieurs personnes ': 'RATE_HOUSESHARE',
    'Ménage comprenant une famille': 'RATE_FAMILY',
    'Famille principale couple sans enfants': 'RATE_FAMILY_CHILDLESS',
    'Famille principale monoparentale': 'RATE_FAMILY_SINGLE_PARENT',
    'Famille principale couple avec enfants': 'RATE_FAMILY_CHILDREN',
    'Total': 'POPULATION_TOTAL'
})

df_household_final = df_household.assign(**{
    col: (df_household[col] / df_household['POPULATION_TOTAL']) * 100
    for col in df_household.columns if col != 'POPULATION_TOTAL'
})

display(df_household_final)

## Merge des différents dataframes

### Merge des dataframes INSEE

In [ ]:
df_insee = pd.concat([df_cities, df_population_final, df_household_final, df_conjugality_final], axis=1, join='inner')

df_insee = df_insee.reset_index()
df_insee = df_insee.sort_values('CODE_INSEE')

display(df_insee)

## Merge avec dataframe elections

### Dataframe complet pour l'analyse

In [ ]:
# Merge initial
df_election = df_election.rename(columns={'Code_INSEE': 'CODE_INSEE'})
display(df_election)

df_merged = df_insee.merge(df_election, on='CODE_INSEE', how='left')


# Récupération des communes et population totale en début de tableau
col_population = df_merged.pop('POPULATION_TOTAL')
col_city_election = df_merged.pop('Libellé de la commune')
df_merged.insert(1, 'CITY_ELECTION', col_city_election)
df_merged.insert(3, 'POPULATION_TOTAL', col_population)


# Nettoyage des valeurs vides et de la colonne city en doublon
df_merged = df_merged.dropna()
df_merged = df_merged.drop('CITY_ELECTION', axis=1)

display(df_merged)


### Dataframe transformé pour le modèle avec la variable cible

In [ ]:
# Transformation des taux de vote en résultat catégoriel (variable cible : gauche, droite ou centre)
df_final = df_merged.copy()

df_final = df_final.rename(columns={'pct_gauche': 'GAUCHE', 'pct_droite': 'DROITE', 'pct_centre': 'CENTRE'})
df_final['RESULT'] = df_final[['GAUCHE', 'DROITE', 'CENTRE']].idxmax(axis=1)
df_final = df_final.drop(['GAUCHE', 'DROITE', 'CENTRE'], axis=1)

col_result = df_final.pop('RESULT')
df_final.insert(2, 'RESULT', col_result)

display(df_final)

## Export des deux dataframes

In [ ]:
df_merged.to_csv('df_analysis.csv', index=False, sep=';', encoding='utf-8')
df_final.to_csv('df_model.csv', index=False, sep=';', encoding='utf-8')
